# Phase 4, Stage 2: Exploratory Plots (rough, for our own understanding)

Before running the formal tests in Stage 3, we want to *see* the shapes we're about to compare. These plots are deliberately rough/working plots, not final report figures (those come in Phase 5).

For each rating band, we'll overlay the `capped_cpl` distribution for all 4 time pressure bins so we can see directly how the shape changes as time pressure increases.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

RATING_BAND_LABELS = {
    1: 'Novice (<1000)',
    2: 'Intermediate (1000-1499)',
    3: 'Club Player (1500-1999)',
    4: 'Advanced (2000-2299)',
    5: 'Expert/Master (2300+)',
}

TIME_PRESSURE_LABELS = {
    1: 'Minimal (>75%)',
    2: 'Low (50-75%)',
    3: 'Moderate (25-50%)',
    4: 'High (<25%)',
}

df = pd.read_csv('../../data/processed/analysed_moves.csv')
print(f'Loaded {len(df):,} rows')

## Full-range overlay: all 4 time pressure bins per rating band

We use `density=True` so each histogram is normalised to its own total (otherwise the bins with fewer moves, like High Pressure, would just look smaller rather than letting us compare *shape*). `histtype='step'` draws outlines instead of filled bars so 4 overlapping distributions stay readable.

In [ ]:
bins = np.arange(0, 310, 10)  # 10cp-wide bins from 0 to 300

fig, axes = plt.subplots(5, 1, figsize=(8, 18), sharex=True)

for ax, rating_band in zip(axes, sorted(RATING_BAND_LABELS)):
    cell = df[df['rating_band'] == rating_band]
    for tp_bin, label in TIME_PRESSURE_LABELS.items():
        values = cell.loc[cell['time_pressure_bin'] == tp_bin, 'capped_cpl']
        ax.hist(values, bins=bins, density=True, histtype='step', linewidth=1.5, label=label)
    ax.set_title(RATING_BAND_LABELS[rating_band])
    ax.set_ylabel('Density (proportion per centipawn)')
    ax.set_xticks(np.arange(0, 310, 50))
    ax.legend(fontsize=8)

axes[-1].set_xlabel('Capped CPL (centipawns)')
fig.suptitle('Capped CPL distribution by time pressure bin, per rating band (full range)', y=1.0)
fig.tight_layout()
fig.savefig('../figures/exploratory_full_range.png', dpi=120, bbox_inches='tight')
plt.show()

## Zoomed view: 0-60 CPL

Most moves sit in the 0-50 range (Inaccuracy/Minor Error territory), so the full-range plot above compresses this region. Zooming in here should make the "median shifts down" pattern from Stage 1 visible directly — i.e. does the bulk of the distribution shift slightly *left* (toward 0) under higher pressure, even while the tail (next plot) grows?

In [ ]:
zoom_bins = np.arange(0, 62, 2)  # 2cp-wide bins from 0 to 60

fig, axes = plt.subplots(5, 1, figsize=(8, 18), sharex=True)

for ax, rating_band in zip(axes, sorted(RATING_BAND_LABELS)):
    cell = df[df['rating_band'] == rating_band]
    for tp_bin, label in TIME_PRESSURE_LABELS.items():
        values = cell.loc[cell['time_pressure_bin'] == tp_bin, 'capped_cpl']
        ax.hist(values, bins=zoom_bins, density=True, histtype='step', linewidth=1.5, label=label)
    ax.set_title(RATING_BAND_LABELS[rating_band])
    ax.set_ylabel('Density (proportion per centipawn)')
    ax.set_xticks(np.arange(0, 62, 10))
    ax.legend(fontsize=8)

axes[-1].set_xlabel('Capped CPL (centipawns)')
fig.suptitle('Capped CPL distribution by time pressure bin, per rating band (0-60 zoom)', y=1.0)
fig.tight_layout()
fig.savefig('../figures/exploratory_zoom_0_60.png', dpi=120, bbox_inches='tight')
plt.show()

## Tail view: log-scale y-axis, full range

A log-scale y-axis stretches out the low-density tail region (50-300, where blunders live), making it easier to compare how "thick" the blunder tail is across time pressure bins — this is the region driving the kurtosis differences we saw in Stage 1.

In [ ]:
fig, axes = plt.subplots(5, 1, figsize=(8, 18), sharex=True)

for ax, rating_band in zip(axes, sorted(RATING_BAND_LABELS)):
    cell = df[df['rating_band'] == rating_band]
    for tp_bin, label in TIME_PRESSURE_LABELS.items():
        values = cell.loc[cell['time_pressure_bin'] == tp_bin, 'capped_cpl']
        ax.hist(values, bins=bins, density=True, histtype='step', linewidth=1.5, label=label)
    ax.set_yscale('log')
    ax.set_title(RATING_BAND_LABELS[rating_band])
    ax.set_ylabel('Density (log scale)')
    ax.set_xticks(np.arange(0, 310, 50))
    ax.legend(fontsize=8)

axes[-1].set_xlabel('Capped CPL (centipawns)')
fig.suptitle('Capped CPL distribution by time pressure bin, per rating band (log-scale density)', y=1.0)
fig.tight_layout()
fig.savefig('../figures/exploratory_log_scale.png', dpi=120, bbox_inches='tight')
plt.show()